# Event Selection

**_must run with dfs with all reconstructed slices and tracks_**

This notebook runs the full event selection, evaluating selection performance:

- mode breakdown bar plots for each selection stage
- purity/efficiency summary plots

## Recommended: live batched workflow (full / partial sample)

When input `.df` files are too large to load at once, set **`USE_BATCHED_WORKFLOW = True`** and run the **Batched workflow** section. That processes files one-by-one (configure **`MAX_FILES_PER_SAMPLE`**), refreshes a large multi-panel of all event-selection overlays as statistics accumulate, then writes `merged_histdata.pkl`.

Non-interactive / cluster equivalent:

- `analysis_village/numucc_1p0pi/scripts/run_event_selection_batched.py`
- or `scripts/run_event_selection_batched.sh`

## Legacy: in-memory workflow

Set **`USE_BATCHED_WORKFLOW = False`** only for small test samples. The cells below load concatenated dataframes into RAM and step through cuts interactively.


In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
# print avaialbe memory
import psutil

vmem = psutil.virtual_memory()
print(f"Available memory: {vmem.available / 1024 ** 3:.2f} GiB / {vmem.total / 1024 ** 3:.2f} GiB total")
print(f"Used: {vmem.used / 1024 ** 3:.2f} GiB ({vmem.percent}%)")

In [ ]:
import pandas as pd
import numpy as np
import sys
from os import path, makedirs
from datetime import datetime
import pickle

# local imports
# sys.path.append('../../../')
sys.path.append('/exp/sbnd/app/users/munjung/xsec/freeze/cafpyana') # absolute path for running on EAF
from analysis_village.numucc_1p0pi.variable_configs import VariableConfig
from analysis_village.numucc_1p0pi.categories import *
from analysis_village.numucc_1p0pi.utils import *
from analysis_village.numucc_1p0pi.files_config import *
from analysis_village.numucc_1p0pi.makedf.selections import *
from pyanalib.split_df_helpers import *
from pyanalib.pandas_helpers import *
from pyanalib.covariance import *

import matplotlib.pyplot as plt 
from matplotlib.patches import Patch

plt.style.use("presentation.mplstyle")

# turn off PerformanceWarning 
# triggered by mismatched column levels
import warnings
warnings.filterwarnings("ignore", category=pd.errors.PerformanceWarning)
# turn off RuntimeWarning
warnings.filterwarnings("ignore", category=RuntimeWarning)

## Configs

In [ ]:
# ===== files =====
base_dir = "/pnfs/sbnd/scratch/users/munjung/cafpyana_out/dfs"

mc_dir = "2026_09_04_172912__sel_all-mc-CV_updated"
mc_dirt_dir = "2026_09_01_135047__sel_all-mc-dirt_updated"
mc_intime_dir = "2026_09_01_135508__sel_all-mc-Intime_updated"
data_dir = "2026_09_04_172930__sel_all-data-1e20_updated"
data_offbeam_dir = "2026_09_01_140024__sel_all-data-OffBeamLight_updated"

import glob
# print how many *df files are in each directory
for this_dir in [mc_dir, mc_dirt_dir, mc_intime_dir, data_dir, data_offbeam_dir]:
    print(f"{this_dir}: {len(glob.glob(f'{base_dir}/{this_dir}/*.df'))}")

In [ ]:
# ===== plot configs =====
from analysis_village.numucc_1p0pi.dataset_locations import PLOTS_BASE

today_str = datetime.now().strftime("%Y%m%d")
today_str = "PRL"
syst_tag = ""

# Optional overrides (None → dated work dir under /exp/sbnd/data/users/$USER/...)
plots_dir = path.join(PLOTS_BASE, f"event_selection-{syst_tag}-{today_str}")

In [ ]:
USE_BATCHED_WORKFLOW = True  # False → legacy in-memory cells below (small tests only)

# Legacy in-memory event selection

**Skip this section when `USE_BATCHED_WORKFLOW=True`** — plots and `merged_histdata.pkl` are already written by the batched workflow above.

Cuts run through a `SampleBundle` (`legacy_samples.py`) so each stage cell only names the cut and its thresholds; the same transform is applied to mc / data / intime / offbeam / dirt automatically.


In [ ]:
if not USE_BATCHED_WORKFLOW:
    from analysis_village.numucc_1p0pi.legacy_samples import SampleBundle

    n_max_files = 10
    samples = SampleBundle.load_from_dirs(
        base_dir,
        {
            "mc":      (mc_dir,           ["hdr", "evt", "trk"]),
            "data":    (data_dir,         ["evt", "trk", "hdr", "bnbpot"]),
            "intime":  (mc_intime_dir,    ["hdr", "evt", "trk"]),
            "offbeam": (data_offbeam_dir, ["hdr", "evt", "trk"]),
            "dirt":    (mc_dirt_dir,      ["hdr", "evt", "trk"]),
        },
        filename_str="sel_all",
        n_max_concat=n_max_files,
    )
else:
    samples = None
    print("Skipping legacy dfs_from_dir load — batched workflow already produced plots.")


In [ ]:
pot_str = getattr(samples, "pot_str", "dummy") if samples is not None else "dummy"
plot_labels_bar = ["Events (POT={})".format(pot_str), "", ""]
plot_labels_hist = ["", "Events (POT={})".format(pot_str), ""]


In [ ]:
if not USE_BATCHED_WORKFLOW:
    syst_tag = ""
    save_fig = True
    save_nevts = True
    show_plot = True

    save_fig_dir = plots_dir

    if save_fig:
        if not path.exists(save_fig_dir):
            makedirs(save_fig_dir)
        print("saving plots in ", save_fig_dir)

    save_nevts_dir = plots_dir
    if save_nevts:
        if not path.exists(save_nevts_dir):
            makedirs(save_nevts_dir)
        print("saving nevts in ", save_nevts_dir)
else:
    save_nevts = False
    print("Using batched plot output:", save_fig_dir)

In [ ]:
# for event selection summary plot
breakdown_dict = {"topology": {}, "genie": {}}
# stage snapshots live on samples.stages / samples.df_dict (MC efficiency)


In [ ]:
def plot_bar_plots(stage_key, mc_df, intime_df, dirt_df, show_plot=True):
    ret_dict = {}
    for bar_type in ["topology", "genie"]:
        save_name = save_fig_dir + "/bar_plot-{}-{}.png".format(bar_type, stage_key)
        this_ret = bar_plot(
            breakdown_type=bar_type,
            mc_df=mc_df, intime_df=intime_df, dirt_df=dirt_df,
            show_plot=show_plot, plot_labels=plot_labels_bar,
            save_fig=save_fig, save_name=save_name,
        )
        ret_dict[bar_type] = this_ret
    return ret_dict


def stage_kwargs():
    """Common kwargs for samples.cut_stage / record_stage bar plots."""
    return dict(
        breakdown_dict=breakdown_dict,
        plot_bar_plots=plot_bar_plots,
        show_plot=show_plot,
    )


In [ ]:
# optional: keep a handle to the pre-cut bundle if you need to re-run stages
# samples_raw = samples  # SampleBundle is mutated in place


In [ ]:
if samples is None:
    raise RuntimeError("Legacy section requires USE_BATCHED_WORKFLOW=False and a loaded SampleBundle")

samples.assign_exposure(f=0.08)
pot_str = samples.pot_str
plot_labels_bar = ["Events (POT={})".format(pot_str), "", ""]
plot_labels_hist = ["", "Events (POT={})".format(pot_str), ""]

# local aliases for interactive inspection / rare one-off plots
mc_df, data_df, intime_df, offbeam_df, dirt_df = samples.unpack_evt()
mc_trk_df, data_trk_df, intime_trk_df, offbeam_trk_df, dirt_trk_df = samples.unpack_trk()
mc_hdr_df, data_hdr_df, intime_hdr_df, offbeam_hdr_df, dirt_hdr_df = (
    samples.hdr[s] for s in ("mc", "data", "intime", "offbeam", "dirt")
)


## Cosmic Rejection

In [ ]:
stage_key = "allreco"
samples.record_stage(stage_key)
# ret = plot_bar_plots(stage_key, **samples.as_overlay_kwargs(include_offbeam=False), show_plot=show_plot)
# for key in ["topology", "genie"]:
#     breakdown_dict[key][stage_key] = ret[key]["perc_list"]


In [ ]:
# pull current evt dfs (same objects as samples.evt)
mc_df, data_df, intime_df, offbeam_df, dirt_df = samples.unpack_evt()


In [ ]:
stage_key = "is_clear_cosmic"
samples.apply_evt(cut_clear_cosmic)
samples.record_stage(stage_key)  # efficiency history only; no bar plot


In [ ]:
samples.refresh_tracks()
samples.attach_prim_trk_cols()  # phi, dir_y, start_x, end_x, P_frac_diff
mc_trk_df, data_trk_df, intime_trk_df, offbeam_trk_df, dirt_trk_df = samples.unpack_trk()
mc_df, data_df, intime_df, offbeam_df, dirt_df = samples.unpack_evt()

In [ ]:
samples.cut_stage(
    "vertex_in_fv",
    cut_vertex_in_fv, det=DETECTOR,
    **stage_kwargs(),
)
mc_df, data_df, intime_df, offbeam_df, dirt_df = samples.unpack_evt()


In [ ]:
var_config = VariableConfig.nu_score()
plot_labels = [var_config.var_labels[0], "Events (POT={})".format(pot_str), ""]
for plot_type in ["topology"]:
    save_name = save_fig_dir + "/selected-{}_{}.png".format(var_config.var_save_name, plot_type)
    ret_hist_topo = overlay_hists(
        plot_type, **samples.as_overlay_kwargs(),
        ratio=True, ax_ylim_ratio=1.7, vline=[[NU_SCORE_TH, 1]],
        var_config=var_config, plot_labels=plot_labels,
        save_fig=save_fig, save_name=save_name,
    )


In [ ]:
samples.apply_evt(cut_nu_score, NU_SCORE_TH)
samples.refresh_tracks(attach_ntrks=True)
samples.record_stage("nu_score", bars=True, **stage_kwargs())
mc_df, data_df, intime_df, offbeam_df, dirt_df = samples.unpack_evt()
mc_trk_df, data_trk_df, intime_trk_df, offbeam_trk_df, dirt_trk_df = samples.unpack_trk()


In [ ]:
plot_type = "topology"
var_config = VariableConfig.prim_direction_phi()
plot_labels = [var_config.var_labels[0], "Events / Bin (POT={})".format(pot_str), ""]
save_name = save_fig_dir + "/2prong-{}_{}.png".format(var_config.var_save_name, plot_type)
ret_hist = overlay_hists(
    plot_type, **samples.as_overlay_kwargs(),
    ratio=True, var_config=var_config, plot_labels=plot_labels,
    save_fig=save_fig, save_name=save_name,
)


## Slice has two tracks

In [ ]:
# TODO: this is for det var samples / move to dfmaker
samples.attach_chi2_avgs()
mc_trk_df, data_trk_df, intime_trk_df, offbeam_trk_df, dirt_trk_df = samples.unpack_trk()


In [ ]:
samples.refresh_tracks(attach_ntrks=True)
mc_df, data_df, intime_df, offbeam_df, dirt_df = samples.unpack_evt()
mc_trk_df, data_trk_df, intime_trk_df, offbeam_trk_df, dirt_trk_df = samples.unpack_trk()


In [ ]:
var_config = VariableConfig.n_trks()
plot_labels = ["Number of tracks", "Events (POT={})".format(pot_str), ""]
for plot_type in ["topology", "genie"]:
    save_name = save_fig_dir + "/selected-{}_{}.png".format("ntrks", plot_type)
    ret_hist_topo = overlay_hists(
        plot_type, **samples.as_overlay_kwargs(),
        ratio=True, ax_ylim_ratio=1.8, var_config=var_config,
        plot_labels=plot_labels, save_fig=save_fig, save_name=save_name,
    )


In [ ]:
samples.cut_stage("2prong", cut_2prong, **stage_kwargs())
mc_df, data_df, intime_df, offbeam_df, dirt_df = samples.unpack_evt()


In [ ]:
samples.cut_stage(
    "2prong-contained",
    cut_2prong_contained, det=DETECTOR,
    **stage_kwargs(),
)
mc_df, data_df, intime_df, offbeam_df, dirt_df = samples.unpack_evt()


In [ ]:
samples.refresh_tracks()
samples.attach_prim_trk_cols(cols=("phi", "start_x", "end_x"))
mc_trk_df, data_trk_df, intime_trk_df, offbeam_trk_df, dirt_trk_df = samples.unpack_trk()
mc_df, data_df, intime_df, offbeam_df, dirt_df = samples.unpack_evt()


In [ ]:
plot_type = "topology"
for var_config in [
    VariableConfig.prim_direction_phi(), VariableConfig.prim_start_x(),
    VariableConfig.prim_end_x(), VariableConfig.vertex_x(),
]:
    plot_labels = [var_config.var_labels[0], "Events / Bin (POT={})".format(pot_str), ""]
    save_name = save_fig_dir + "/2prong-{}_{}.png".format(var_config.var_save_name, plot_type)
    ret_hist = overlay_hists(
        plot_type, **samples.as_overlay_kwargs(),
        ratio=True, var_config=var_config, plot_labels=plot_labels,
        save_fig=save_fig, save_name=save_name,
    )


In [ ]:
plot_type = "pdg"
samples.set_trk_from_concat()
var_config = VariableConfig.track_score()
plot_labels = [var_config.var_labels[0], "Tracks / Bin (POT={})".format(pot_str), ""]
save_name = save_fig_dir + "/2prong-{}_{}.png".format(var_config.var_save_name, plot_type)
ret_hist = overlay_hists(
    plot_type, **samples.as_overlay_kwargs("trk"),
    ratio=True, ax_ylim_ratio=1.8, vline=[[TRACKSCORE_TH, 1]],
    var_config=var_config, plot_labels=plot_labels,
    save_fig=save_fig, save_name=save_name,
)
with open(f'{save_nevts_dir}/{syst_tag}_{var_config.var_save_name}.pkl', 'wb') as f:
    pickle.dump(ret_hist, f)


In [ ]:
samples.cut_stage(
    "2prong-trackscore",
    cut_2prong_trackscore, TRACKSCORE_TH,
    **stage_kwargs(),
)
mc_df, data_df, intime_df, offbeam_df, dirt_df = samples.unpack_evt()


In [ ]:
plot_type = "pdg"
var_config = VariableConfig.trk_direction_phi()
plot_labels = [var_config.var_labels[0], "Events / Bin (POT={})".format(pot_str), ""]
save_name = save_fig_dir + "/2prong-{}_{}.png".format(var_config.var_save_name, plot_type)
ret_hist = overlay_hists(
    plot_type, **samples.as_overlay_kwargs(selector=lambda df: df.trk2),
    ratio=True, var_config=var_config, plot_labels=plot_labels,
    save_fig=save_fig, save_name=save_name,
)


In [ ]:
plot_type = "pdg"
samples.set_trk_from_concat()
var_config = VariableConfig.vtx_dist()
plot_labels = [var_config.var_labels[0], "Tracks / Bin (POT={})".format(pot_str), ""]
save_name = save_fig_dir + "/2prong-{}_{}.png".format(var_config.var_save_name, plot_type)
ret_hist = overlay_hists(
    plot_type, **samples.as_overlay_kwargs("trk"),
    ratio=True, vline=[[VTXDIST_TH, 0]],
    var_config=var_config, plot_labels=plot_labels,
    save_fig=save_fig, save_name=save_name,
)
with open(f'{save_nevts_dir}/{syst_tag}_{var_config.var_save_name}.pkl', 'wb') as f:
    pickle.dump(ret_hist, f)


In [ ]:
plot_type = "pdg"
var_config = VariableConfig.trk_direction_phi()
plot_labels = [var_config.var_labels[0], "Events / Bin (POT={})".format(pot_str), ""]
save_name = save_fig_dir + "/2prong-{}_{}.png".format(var_config.var_save_name, plot_type)
ret_hist = overlay_hists(
    plot_type, **samples.as_overlay_kwargs(selector=lambda df: df.trk1),
    ratio=True, var_config=var_config, plot_labels=plot_labels,
    save_fig=save_fig, save_name=save_name,
)


In [ ]:
samples.cut_stage(
    "2prong-vtxdist",
    cut_2prong_vtxdist, VTXDIST_TH,
    **stage_kwargs(),
)
mc_df, data_df, intime_df, offbeam_df, dirt_df = samples.unpack_evt()


In [ ]:
plot_type = "pdg"
var_config = VariableConfig.trk_direction_phi()
plot_labels = [var_config.var_labels[0], "Events / Bin (POT={})".format(pot_str), ""]
save_name = save_fig_dir + "/2prong-{}_{}.png".format(var_config.var_save_name, plot_type)
ret_hist = overlay_hists(
    plot_type, **samples.as_overlay_kwargs(selector=lambda df: df.trk1),
    ratio=True, var_config=var_config, plot_labels=plot_labels,
    save_fig=save_fig, save_name=save_name,
)


## PID


In [ ]:
samples.set_trk_from_concat()
samples.attach_chi2_avgs()  # ensure avg columns present for PID plots
mc_trk_df = samples.trk["mc"]

# compare using avg vs. single plane (axis labels match PID: plane-averaged χ²)
plabels = ["Plane 0", "Plane 1", "Plane 2", "Average"]

var_config = VariableConfig.chi2_avg_mu()
for ptag, plabel in zip(["I0", "I1", "I2", "avg"], plabels):
    var = mc_trk_df[("pfp", "trk", "chi2pid", ptag, "chi2_muon", "")]
    plt.hist(var, bins=var_config.bins, histtype="step", label=plabel, linewidth=2)

    plt.xlim(var_config.bins[0], var_config.bins[-1])
    plt.xlabel(var_config.var_labels[0])
    plt.ylabel("Tracks / Bin")
    plt.legend()
    plt.title("Tracks in 2-track Slices")

    # percentage of tracks with score = 0
    print(f"Percentage of tracks with score = 0: {np.sum((var == 0) | (var == np.nan)) / len(var)}")

if save_fig:
    plt.savefig(save_fig_dir + f"/{var_config.var_save_name}_plane_comparison.pdf", bbox_inches="tight")
    plt.show()

var_config = VariableConfig.chi2_avg_proton()
for ptag, plabel in zip(["I0", "I1", "I2", "avg"], plabels):
    var = mc_trk_df[("pfp", "trk", "chi2pid", ptag, "chi2_proton", "")]
    plt.hist(var, bins=var_config.bins, histtype="step", label=plabel, linewidth=2)

    plt.xlim(var_config.bins[0], var_config.bins[-1])
    plt.xlabel(var_config.var_labels[0])
    plt.ylabel("Tracks / Bin")
    plt.legend()
    plt.title("Tracks in 2-track Slices")

if save_fig:
    plt.savefig(save_fig_dir + f"/{var_config.var_save_name}_plane_comparison.pdf", bbox_inches="tight")
    plt.show()

var_config = VariableConfig.chi2_avg_proton()
for ptag, plabel in zip(["I0", "I1", "I2", "avg"], plabels):
    var = mc_trk_df[("pfp", "trk", "chi2pid", ptag, "chi2_proton", "")]
    bins = np.linspace(0, 10, 40)
    plt.hist(var, bins=bins, histtype="step", label=plabel, linewidth=2)
    plt.xlim(bins[0], bins[-1])
    plt.xlabel(var_config.var_labels[0])
    plt.ylabel("Tracks / Bin")
    plt.legend()
    plt.title("Tracks in 2-track Slices")

if save_fig:
    plt.savefig(save_fig_dir + f"/{var_config.var_save_name}_plane_comparison_zoom.pdf", bbox_inches="tight")
    plt.show()


In [ ]:
# does this make the behaviour more isotropic?
var_config = VariableConfig.chi2_mu()
var1 = mc_trk_df[("pfp", "trk", "chi2pid", "I2", "chi2_muon", "")]
bins1 = var_config.bins

var2 = mc_trk_df[("pfp", "trk", "dir", "z", "", "")]
bins2 = np.linspace(0, 1, 20)
plt.hist2d(var1, var2, bins=[bins1, bins2])
plt.colorbar(label="Tracks")
plt.xlabel(var_config.var_labels[0])
plt.ylabel("Z-Direction")
plt.show()

var_config = VariableConfig.chi2_avg_mu()
var1 = mc_trk_df[("pfp", "trk", "chi2pid", "avg", "chi2_muon", "")]
bins1 = var_config.bins

var2 = mc_trk_df[("pfp", "trk", "dir", "z", "", "")]
bins2 = np.linspace(0, 1, 20)
plt.hist2d(var1, var2, bins=[bins1, bins2])
plt.colorbar(label="Tracks")
plt.xlabel(var_config.var_labels[0])
plt.ylabel("Z-Direction")
plt.show()


In [ ]:
# all tracks in 2-track slices
plot_type = "pdg"
samples.set_trk_from_concat()
mc_trk_df, data_trk_df, intime_trk_df, offbeam_trk_df, dirt_trk_df = samples.unpack_trk()

for var_config in [VariableConfig.trk_len()]:
    plot_labels = [var_config.var_labels[0], "Tracks / Bin (POT={})".format(pot_str), ""]
    save_name = save_fig_dir + "/2prong-{}_{}.png".format(var_config.var_save_name, plot_type)
    ret_hist = overlay_hists(
        plot_type, **samples.as_overlay_kwargs("trk"),
        ratio=True, vline=[[50, 1]],
        var_config=var_config, plot_labels=plot_labels,
        save_fig=save_fig, save_name=save_name,
    )
    with open(f'{save_nevts_dir}/{syst_tag}_{var_config.var_save_name}.pkl', 'wb') as f:
        pickle.dump(ret_hist, f)


In [ ]:
samples.attach_mcs_range_diff()
mc_trk_df, data_trk_df, intime_trk_df, offbeam_trk_df, dirt_trk_df = samples.unpack_trk()


In [ ]:
# tracks after len > 50 cm cut for muon selection
samples.apply_trk(lambda trks: trks[trks.pfp.trk.len > MU_LEN_TH])
mc_trk_df, data_trk_df, intime_trk_df, offbeam_trk_df, dirt_trk_df = samples.unpack_trk()

for var_config in [VariableConfig.mcs_range_diff()]:
    plot_labels = [var_config.var_labels[0], "Tracks / Bin (POT={})".format(pot_str), ""]
    save_name = save_fig_dir + "/2prong-{}_{}.png".format(var_config.var_save_name, plot_type)
    ret_hist_chi2mu = overlay_hists(
        plot_type, **samples.as_overlay_kwargs("trk"),
        syst=None, ratio=True, vline=[[-QUAL_TH, 0], [QUAL_TH, 1]],
        var_config=var_config, plot_labels=plot_labels,
        save_fig=save_fig, save_name=save_name,
    )
    with open(f'{save_nevts_dir}/{syst_tag}_{var_config.var_save_name}-len50cm.pkl', 'wb') as f:
        pickle.dump(ret_hist_chi2mu, f)


In [ ]:
# tracks after quality cut for muon selection (plots use current len-cut sample)
# samples.apply_trk(lambda trks: trks[trks.pfp.trk.mcs_range_diff < QUAL_TH])

# PID cuts use plane-averaged χ² — plot/label the avg, not I2.
var_config = VariableConfig.chi2_avg_mu()
plot_labels = [var_config.var_labels[0], "Tracks/ Bin  (POT={})".format(pot_str), ""]
save_name = save_fig_dir + "/2prong-{}_{}.png".format(var_config.var_save_name, plot_type)
ret_hist_chi2mu = overlay_hists(
    plot_type, **samples.as_overlay_kwargs("trk"),
    syst=None, ratio=True, vline=[[MU_CHI2MU_TH, 0]],
    var_config=var_config, plot_labels=plot_labels,
    save_fig=save_fig, save_name=save_name,
)
with open(f'{save_nevts_dir}/{syst_tag}_{var_config.var_save_name}-len50cm_qual{QUAL_TH}.pkl', 'wb') as f:
    pickle.dump(ret_hist_chi2mu, f)

var_config = VariableConfig.chi2_avg_proton()
plot_labels = [var_config.var_labels[0], "Tracks / Bin (POT={})".format(pot_str), ""]
save_name = save_fig_dir + "/2prong-{}_{}.png".format(var_config.var_save_name, plot_type)
ret_hist_chi2p = overlay_hists(
    plot_type, **samples.as_overlay_kwargs("trk"),
    syst=None, ratio=True, vline=[[MU_CHI2P_TH, 1]], ax_ylim_ratio=1.8,
    var_config=var_config, plot_labels=plot_labels,
    save_fig=save_fig, save_name=save_name,
)
with open(f'{save_nevts_dir}/{syst_tag}_{var_config.var_save_name}-len50cm_qual{QUAL_TH}.pkl', 'wb') as f:
    pickle.dump(ret_hist_chi2p, f)


In [ ]:
# get the percentage of tracks selected as mu
def get_mu_cut(trks):
    chimu_avg = trks.pfp.trk.chi2pid.avg.chi2_muon
    chip_avg = trks.pfp.trk.chi2pid.avg.chi2_proton
    return (chimu_avg > 0) & (chimu_avg < MU_CHI2MU_TH) & (chip_avg > MU_CHI2P_TH)

mu_trks = {s: samples.trk[s][get_mu_cut(samples.trk[s])] for s in samples.trk}
n_mu = {s: len(mu_trks[s].groupby(level=[0, 1, 2]).head(1)) for s in mu_trks}
n_tot = {s: len(samples.trk[s]) for s in samples.trk}

print((n_mu["mc"] + n_mu["intime"] + n_mu["dirt"]) /
      (n_tot["mc"] + n_tot["intime"] + n_tot["dirt"] + n_tot["offbeam"]))
print(n_mu["data"] / n_tot["data"])


In [ ]:
# compare chi2_avg_mu vs. chi2_avg_p 2D distributions (same quantities as PID cuts)
from matplotlib.colors import LogNorm

var_config_1 = VariableConfig.chi2_avg_mu()
var_config_2 = VariableConfig.chi2_avg_proton()

for tag, trk_df, label in (
    ("MC", samples.trk["mc"], r"$\mathbf{SBND}$ Internal    $\mathbf{SBND}$ Simulation"),
    ("data", samples.trk["data"], r"$\mathbf{SBND}$ Internal    $\mathbf{SBND}$ Data"),
):
    plt.hist2d(
        trk_df[var_config_1.var_evt_reco_col],
        trk_df[var_config_2.var_evt_reco_col],
        bins=[var_config_1.bins, var_config_2.bins],
        cmap="GnBu", norm=LogNorm(),
    )
    plt.colorbar(label="Tracks")
    plt.xlabel(var_config_1.var_labels[0])
    plt.ylabel(var_config_2.var_labels[0])
    plt.plot([MU_CHI2MU_TH, MU_CHI2MU_TH], [MU_CHI2P_TH, var_config_2.bins[-1]], color="red", linestyle="--")
    plt.plot([0, MU_CHI2MU_TH], [MU_CHI2P_TH, MU_CHI2P_TH], color="red", linestyle="--")
    plt.text(0.025, 1.07, label, transform=plt.gca().transAxes, fontsize=14, color="rosybrown", ha="left", va="top")
    if save_fig:
        plt.savefig(save_fig_dir + f"/chi2avg_muon_vs_chi2avg_proton-{tag}.pdf", bbox_inches="tight")
    plt.show()


In [ ]:
# tracks that aren't muon candidates
plot_type = "pdg"

def is_not_mu_candidate(trks):
    nlevels = len(trks.index.names) - 1  # event levels
    mcs_range_diff = np.abs(
        (trks.pfp.trk.rangeP.p_muon - trks.pfp.trk.mcsP.fwdP_muon) / trks.pfp.trk.rangeP.p_muon
    )
    chimu_avg = trks.pfp.trk.chi2pid.avg.chi2_muon
    chip_avg = trks.pfp.trk.chi2pid.avg.chi2_proton
    mu_cut = (
        (chimu_avg > 0) & (chimu_avg < MU_CHI2MU_TH) &
        (chip_avg > MU_CHI2P_TH) &
        (trks.pfp.trk.len > MU_LEN_TH) &
        (mcs_range_diff < QUAL_TH)
    )
    return pd.concat([trks[~mu_cut], trks[mu_cut].groupby(level=list(range(nlevels))).nth(1)])

samples.apply_trk(is_not_mu_candidate)

for var_config in [VariableConfig.chi2_avg_proton()]:
    plot_labels = [var_config.var_labels[0], "Events (POT={})".format(pot_str), ""]
    save_name = save_fig_dir + "/2prong-{}_{}.png".format(var_config.var_save_name, plot_type)
    ret_hist = overlay_hists(
        plot_type, **samples.as_overlay_kwargs("trk"),
        syst=None, ratio=True, vline=[[P_CHI2P_TH, 0]],
        var_config=var_config, plot_labels=plot_labels,
        save_fig=save_fig, save_name=save_name,
    )
    with open(f'{save_nevts_dir}/{syst_tag}_{var_config.var_save_name}-not_mu_candidate.pkl', 'wb') as f:
        pickle.dump(ret_hist, f)


In [ ]:
var_config_1 = VariableConfig.chi2_avg_mu()
var_config_2 = VariableConfig.chi2_avg_proton()

for tag, trk_df, label in (
    ("MC", samples.trk["mc"], r"$\mathbf{SBND}$ Internal    $\mathbf{SBND}$ Simulation"),
    ("data", samples.trk["data"], r"$\mathbf{SBND}$ Internal    $\mathbf{SBND}$ Data"),
):
    plt.hist2d(
        trk_df[var_config_1.var_evt_reco_col],
        trk_df[var_config_2.var_evt_reco_col],
        bins=[var_config_1.bins, var_config_2.bins],
        cmap="GnBu", norm=LogNorm(),
    )
    plt.colorbar(label="Tracks")
    plt.xlabel(var_config_1.var_labels[0])
    plt.ylabel(var_config_2.var_labels[0])
    plt.axhline(P_CHI2P_TH, color="red", linestyle="--")
    plt.text(0.025, 1.07, label, transform=plt.gca().transAxes, fontsize=14, color="rosybrown", ha="left", va="top")
    if save_fig:
        plt.savefig(save_fig_dir + f"/chi2avg_muon_vs_chi2avg_proton-not_mu_candidate-{tag}.pdf", bbox_inches="tight")
    plt.show()


In [ ]:
samples.apply_evt(
    get_mu_p_candidate,
    mu_chi2mu_th=MU_CHI2MU_TH, mu_chi2p_th=MU_CHI2P_TH, mu_len_th=MU_LEN_TH, qual_th=QUAL_TH,
    p_chi2mu_th=-1, p_chi2p_th=P_CHI2P_TH, p_len_th=P_LEN_TH,
)
mc_df, data_df, intime_df, offbeam_df, dirt_df = samples.unpack_evt()


In [ ]:
samples.apply_evt(cut_has_mu)
samples.apply_evt(cut_mu_kinematics, mu_Plo_th=MU_PLO_TH, mu_Phi_th=MU_PHI_TH)
samples.record_stage("2prong-muX", bars=True, **stage_kwargs())
mc_df, data_df, intime_df, offbeam_df, dirt_df = samples.unpack_evt()


In [ ]:
samples.apply_evt(cut_has_p)
samples.apply_evt(cut_p_kinematics, p_Plo_th=P_PLO_TH, p_Phi_th=P_PHI_TH)
samples.record_stage("2prong-mup", bars=True, **stage_kwargs())
mc_df, data_df, intime_df, offbeam_df, dirt_df = samples.unpack_evt()


In [ ]:
# per-TPC containment (same logic as the 2prong-contained stage)
perTPC_cut = event_contained_per_tpc(samples.evt["mc"])
print("precut", len(samples.evt["mc"]))
print("postcut", len(samples.evt["mc"][perTPC_cut]))
print("eff", len(samples.evt["mc"][perTPC_cut]) / len(samples.evt["mc"]))


In [ ]:
for k in samples.df_dict_data.keys():
    print(k, len(samples.df_dict_data[k]))


In [ ]:
sel_dfs = {
    "mc_df": samples.evt["mc"],
    "data_df": samples.evt["data"],
    "intime_df": samples.evt["intime"],
    "dirt_df": samples.evt["dirt"],
}
with open(f'{save_nevts_dir}/{syst_tag}_sel_mup.pkl', 'wb') as f:
    pickle.dump(sel_dfs, f)


In [ ]:
save_nevts_dir

# Batched workflow (recommended for full samples)

Uses the **same `base_dir` / sample directories** as the notebook config cell above
(`mc_dir`, `data_dir`, …). Processes files one-by-one, aggregates after each round,
and redraws a live multi-panel of every pipeline event-selection overlay.

For a headless full-sample run without live plots, use
`scripts/run_event_selection_batched.py` (or the `.sh` wrapper).


In [ ]:
# --- live / partial-run knobs ---
MAX_FILES_PER_SAMPLE = None  # int, or None for all files
UPDATE_EVERY_N_FILES = 20  # refresh inline panel every N files (also at the end)
FILES_PER_JOB = None         # None → same as UPDATE_EVERY_N_FILES (concat load)
FILENAME_STR = "sel_all"     # same filter as legacy SampleBundle.load_from_dirs
SHOW_IN_NOTEBOOK = True      # live panel via IPython display (not a file)
SAVE_MERGED_PAYLOAD = True   # write merged_histdata.pkl (reloadable counts)
SAVE_FINAL_PANEL = False     # optional PNG under plots_dir
INCLUDE_EFFICIENCY = True    # append efficiency curves to the live panel
COSMIC_ESTIMATE = "intime"   # "intime" | "offbeam" — cosmic sample in live panel + follow-up plots

# Same layout as the files config cell / legacy load
BATCH_SAMPLE_DIRS = {
    "mc":      mc_dir,
    "data":    data_dir,
    "intime":  mc_intime_dir,
    "offbeam": data_offbeam_dir,
    "dirt":    mc_dirt_dir,
}

batch_work_base = None     # None → default under /exp/sbnd/data/users/$USER/...
batch_plots_dir = plots_dir


In [ ]:
from analysis_village.numucc_1p0pi.event_selection_live import (
    LiveAccumulateConfig,
    run_live_accumulate,
    replot_from_payload,
)
from analysis_village.numucc_1p0pi.event_selection_batched import (
    EventSelectionBatchedConfig,
    discover_jobs,
    run_full,
    show_saved_plots,
    default_event_selection_batched_work_root,
)

live_cfg = LiveAccumulateConfig(
    max_files_per_sample=MAX_FILES_PER_SAMPLE,
    files_per_job=FILES_PER_JOB,
    one_file_per_job=False,
    concat_load=True,
    update_every_round=True,
    update_every_n_files=UPDATE_EVERY_N_FILES,
    base_dir=base_dir,
    sample_dirs=BATCH_SAMPLE_DIRS,
    filename_str=FILENAME_STR,
    work_base=batch_work_base or default_event_selection_batched_work_root(
        f"live-{today_str}"
    ),
    plots_dir=batch_plots_dir,
    mc_univ_syst=("Flux", "G4", "GENIE"),
    skip_existing=False,
    show_in_notebook=SHOW_IN_NOTEBOOK,
    include_efficiency=INCLUDE_EFFICIENCY,
    save_merged_payload=SAVE_MERGED_PAYLOAD,
    save_final_panel=SAVE_FINAL_PANEL,
    cosmic_estimate=COSMIC_ESTIMATE,
    figsize=(28, 40),
    ncols=5,
)

# Non-interactive fallback (same as scripts/run_event_selection_batched.py)
batch_cfg = EventSelectionBatchedConfig(
    work_base=batch_work_base or default_event_selection_batched_work_root(today_str),
    plots_dir=batch_plots_dir,
    max_job_bytes=int(1.0 * 1024**3),
    max_files_per_sample=MAX_FILES_PER_SAMPLE,
    base_dir=base_dir,
    sample_dirs=BATCH_SAMPLE_DIRS,
    filename_str=FILENAME_STR,
    mc_univ_syst=("Flux", "G4", "GENIE"),
    skip_existing_batches=True,
    aggregate_only=False,
    skip_aggregate=False,
    save_fig=True,
    show_fig=False,
    cosmic_estimate=COSMIC_ESTIMATE,
)

if USE_BATCHED_WORKFLOW:
    print(
        f"Live batched config: max_files_per_sample={MAX_FILES_PER_SAMPLE}  "
        f"update_every_n_files={UPDATE_EVERY_N_FILES}  "
        f"base_dir={base_dir}  work={live_cfg.work_base}"
    )
    for s, d in BATCH_SAMPLE_DIRS.items():
        print(f"  {s}: {d}")
    assert live_cfg.base_dir and live_cfg.sample_dirs, (
        "live_cfg missing base_dir/sample_dirs — re-run the knobs cell"
    )


In [ ]:
batch_result = None
live_result = None
merged_payload = None

if USE_BATCHED_WORKFLOW:
    # Interactive path: accumulate stats file-by-file and refresh the big panel.
    # Skip this cell if you already have merged_histdata.pkl — the follow-up
    # section can reload it via MERGED_HISTDATA_PKL / plots_dir.
    live_result = run_live_accumulate(live_cfg)

    save_fig_dir = str(live_result.plots_dir)
    save_fig = True
    show_plot = True
    pot_str = live_result.pot_str
    data_tot_pot = live_result.data_pot
    merged_payload = live_result.merged_payload

    print("Live batched workflow complete.")
    print("  batches :", live_result.batches_dir)
    print("  plots   :", live_result.plots_dir)
    print("  payload :", live_result.payload_path)
    print("  panel   :", live_result.panel_path)
    print("  files   :", live_result.n_files_done)
    print("  POT     :", pot_str)
    if live_result.failed:
        print(f"  FAILED  : {len(live_result.failed)} job(s)")
else:
    print("USE_BATCHED_WORKFLOW=False — run legacy data-loading cells below.")


# Summary, overlays & efficiency

**Batched (`USE_BATCHED_WORKFLOW=True`):** load `merged_histdata.pkl` from the live run (or set `MERGED_HISTDATA_PKL`) and re-render plots. Toggle cosmics with `COSMIC_ESTIMATE = "intime"` or `"offbeam"`.

**Legacy:** uses in-memory `samples` / `breakdown_dict` / `df_dict` from the cells above.


In [ ]:
# --- follow-up knobs (batched) ---
# Reload without re-running the live cell: set a path, or leave None to use
# in-memory merged_payload / {plots_dir}/merged_histdata.pkl
MERGED_HISTDATA_PKL = None  # e.g. path.join(plots_dir, "merged_histdata.pkl")

# Override cosmic estimate here if you want offbeam without re-editing the knobs cell
# COSMIC_ESTIMATE = "offbeam"

STAGE_LABELS = {
    "allreco": "All reconstructed slices",
    "is_clear_cosmic": "Not clear cosmic",
    "vertex_in_fv": "Vertex in Gen-1 fiducial volume",
    "nu_score": "Nu-score > {}".format(NU_SCORE_TH),
    "2prong": "Has exactly 2 PFPs",
    "2prong-contained": "Both PFPs per-TPC contained",
    "2prong-trackscore": "Both PFPs have track score > {}".format(TRACKSCORE_TH),
    "2prong-vtxdist": "Both track \n(start position - vertex) < {} cm".format(VTXDIST_TH),
    "2prong-muX": "One track is muon-like",
    "2prong-mup": "The other is proton-like",
}

if USE_BATCHED_WORKFLOW:
    from analysis_village.numucc_1p0pi.event_selection_live import (
        resolve_merged_payload,
        render_followup_from_payload,
        replot_from_payload,
    )

    _mp = globals().get("merged_payload")
    _lr = globals().get("live_result")
    merged_payload, payload_path = resolve_merged_payload(
        merged_payload=_mp,
        live_result=_lr,
        plots_dir=globals().get("save_fig_dir") or plots_dir,
        merged_histdata_pkl=MERGED_HISTDATA_PKL,
    )
    save_fig_dir = str(
        getattr(_lr, "plots_dir", None)
        or (payload_path.parent if payload_path.suffix == ".pkl" else None)
        or plots_dir
    )
    pot_str = merged_payload.get("pot_str") or globals().get("pot_str", "dummy")
    save_fig = True
    show_plot = True
    approval = globals().get("approval", "")
    print(f"Follow-up from: {payload_path}")
    print(f"  save_fig_dir     = {save_fig_dir}")
    print(f"  pot_str          = {pot_str}")
    print(f"  COSMIC_ESTIMATE  = {COSMIC_ESTIMATE}")
    print(f"  n_files_done     = {merged_payload.get('n_files_done')}")
    print(f"  histdata plots   = {len(merged_payload['merged'].get('histdata', {}))}")
    print(f"  bar stages       = {sorted(merged_payload['merged'].get('bar', {}))}")
else:
    df_dict = samples.df_dict  # MC stage history for efficiency
    print(list(df_dict.keys()))
    stage_labels = [STAGE_LABELS[k] for k in df_dict.keys()]


In [ ]:
if USE_BATCHED_WORKFLOW:
    followup = render_followup_from_payload(
        merged_payload,
        save_fig_dir=save_fig_dir,
        cosmic_estimate=COSMIC_ESTIMATE,
        save_fig=save_fig,
        show_fig=show_plot,
        render_overlays=True,
        render_summary=True,
        render_efficiency=True,
    )
    print("Wrote follow-up plots under", followup["save_fig_dir"])
else:
    stages = list(breakdown_dict["topology"].keys())[::-1]
    stage_tick_labels = [STAGE_LABELS[k] for k in stages]
    y = np.arange(len(stages))
    bar_width = 0.3

    def stack_bars(ax, data, yoffset, colors, label):
        left = np.zeros(len(stages))
        bars = []
        for i, color in enumerate(colors[:data.shape[1]]):
            b = ax.barh(y + yoffset, data[:, i], bar_width, left=left, color=color, label=label if i == 0 else None)
            bars.append(b)
            left += data[:, i]
        return bars

    topo_data = np.array([breakdown_dict["topology"][stage] for stage in stages])[:, ::-1]
    genie_data = np.array([breakdown_dict["genie"][stage] for stage in stages])[:, ::-1]

    fig, ax = plt.subplots(figsize=(10, 10))
    stack_bars(ax, topo_data, -bar_width/2, topology_colors, "Topology")
    stack_bars(ax, genie_data,  bar_width/2,  genie_mode_colors, "GENIE")

    ax.set_xlabel("Percentage (%)")
    ax.set_yticks(y)
    ax.set_yticklabels(stage_tick_labels, fontsize=12)

    common_patches = [Patch(facecolor=c, label=l) for c, l in zip(
        ["gray", "sienna", "crimson", "darkgreen"],
        ["Cosmic", r"Out FV $\nu$", r"In FV other $\nu$", r"In FV $\nu_{\mu}$ NC"]
    )]
    genie_patches = [Patch(facecolor=c, label=l) for c, l in zip(
        ["#BFB17C", "#D88A3B", "#2c7c94", "#390C1E", "#9b5580"],
        [r"In FV $\nu_{\mu}$ CC Other", r"In FV $\nu_{\mu}$ CC SIS/DIS", r"In FV $\nu_{\mu}$ CC RES", r"In FV $\nu_{\mu}$ CC MEC", r"In FV $\nu_{\mu}$ CC QE"]
    )]
    topo_patches = [Patch(facecolor=c, label=l) for c, l in zip(
        ["coral", "darkslateblue", "mediumslateblue"],
        [r"In FV $\nu_{\mu}$ CC Other", r"In FV $\nu_{\mu}$ CC Np0$\pi$", r"In FV $\nu_{\mu}$ CC 1p0$\pi$"]
    )]

    ax.legend(handles=common_patches, loc='upper left', bbox_to_anchor=(0.01,1.18), ncol=4, fontsize=12, frameon=False)
    for i, handles in enumerate([genie_patches[::-1], topo_patches[::-1]]):
        ax_i = ax.twinx()
        ax_i.legend(handles=handles, loc='upper left',
                    bbox_to_anchor=(0.01, 1.14 - 0.07*i),
                    ncol=3 if i==0 else 4, fontsize=12, frameon=False)
        ax_i.set_yticks([])

    if save_fig:
        plt.savefig(f"{save_fig_dir}/event_selection_summary.png", dpi=300, bbox_inches="tight")

    plt.tight_layout()
    plt.show()


In [ ]:
# Optional: rebuild the multi-panel live figure from the payload (no reprocessing)
if USE_BATCHED_WORKFLOW:
    fig = replot_from_payload(merged_payload, cosmic_estimate=COSMIC_ESTIMATE)
    plt.show()
else:
    save_fig_dir


In [ ]:
# Legacy-only quick overlay check (batched path already rendered all overlays above)
if not USE_BATCHED_WORKFLOW:
    eps = 1e-8
    ratio = True
    approval = ""
    textloc = [0.05, 0.55]
    ax_ylim_ratio = 1.6
    breakdown_type = "topology"

    ret = overlay_hists(
        breakdown_type=breakdown_type,
        var_config=VariableConfig.muon_momentum(),
        **samples.as_overlay_kwargs(),
        ax_ylim_ratio=ax_ylim_ratio,
        ratio=ratio,
        textloc=textloc,
        approval=approval,
        plot_labels=plot_labels_hist,
        syst=None,
        save_fig=False,
        save_name=None,
    )


# Efficiency Curves

Batched mode already wrote `efficiency-*.png` and `eff_dict.pkl` in the follow-up cell.
The cells below are **legacy-only** (`USE_BATCHED_WORKFLOW=False`).


In [ ]:
if USE_BATCHED_WORKFLOW:
    print(f"Efficiency plots already under {save_fig_dir} (see follow-up cell).")
else:
    eff_dict = {}
    for var_config in [VariableConfig.muon_momentum(), VariableConfig.muon_direction(),
                       VariableConfig.proton_momentum(), VariableConfig.proton_direction()]:

       save_name = save_fig_dir + "/efficiency-{}.png".format(var_config.var_save_name)
       ret = plot_efficiency(df_dict,
                       stage_labels,
                       var_config,
                       textloc=[0.05, 1.08],
                       approval=approval,
                       legend=False,
                       save_fig=save_fig,
                       save_name=save_name)

       eff_dict[var_config.var_save_name] = ret


In [ ]:
if not USE_BATCHED_WORKFLOW:
    for var_config in [VariableConfig.muon_direction_x(), VariableConfig.muon_direction_y(),
                       VariableConfig.proton_direction_x(), VariableConfig.proton_direction_y()]:

       save_name = save_fig_dir + "/efficiency-{}.png".format(var_config.var_save_name)
       ret = plot_efficiency(df_dict,
                       stage_labels,
                       var_config,
                       textloc=[0.05, 1.08],
                       approval=approval,
                       legend=False,
                       save_fig=save_fig,
                       save_name=save_name)

       eff_dict[var_config.var_save_name] = ret


In [ ]:
if not USE_BATCHED_WORKFLOW:
    for var_config in [ VariableConfig.vertex_x(), VariableConfig.vertex_y(), VariableConfig.vertex_z()]:

       save_name = save_fig_dir + "/efficiency-{}.png".format(var_config.var_save_name)
       ret = plot_efficiency(df_dict,
                       stage_labels,
                       var_config,
                       textloc=[0.05, 1.08],
                       approval=approval,
                       legend=False,
                       save_fig=save_fig,
                       save_name=save_name)

       eff_dict[var_config.var_save_name] = ret


In [ ]:
if not USE_BATCHED_WORKFLOW:
    for var_config in [ VariableConfig.opening_angle(),
                       VariableConfig.tki_del_Tp(), VariableConfig.tki_del_p(), VariableConfig.tki_del_Tp_x(), VariableConfig.tki_del_Tp_y(),
                       VariableConfig.tki_del_alpha(), VariableConfig.tki_del_phi(),]:

       save_name = save_fig_dir + "/efficiency-{}.png".format(var_config.var_save_name)
       ret = plot_efficiency(df_dict,
                       stage_labels,
                       var_config,
                       textloc=[0.05, 1.08],
                       approval=approval,
                       legend=False,
                       save_fig=save_fig,
                       save_name=save_name)

       eff_dict[var_config.var_save_name] = ret


In [ ]:
if not USE_BATCHED_WORKFLOW:
    var_config = VariableConfig.neutrino_energy()
    save_name = save_fig_dir + "/efficiency-{}.png".format(var_config.var_save_name)
    ret = plot_efficiency(df_dict,
                    stage_labels,
                    var_config,
                    textloc=[0.05, 1.08],
                    approval=approval,
                    legend=True,
                    save_fig=save_fig,
                    save_name=save_name)

    eff_dict[var_config.var_save_name] = ret


In [ ]:
if USE_BATCHED_WORKFLOW:
    print(f"Batched eff_dict already at {path.join(save_fig_dir, 'eff_dict.pkl')}")
else:
    with open(f'{save_fig_dir}/{syst_tag}_eff_dict.pkl', 'wb') as f:
        pickle.dump(eff_dict, f)
